# 01 · WM-811K exploratory analysis

**Goal:** understand the wafer-map data before building a defect-pattern classifier, and check whether defect patterns relate to die yield (the link used in step 4).

**Questions**
1. How many wafers are labeled, and how imbalanced are the 9 classes?
2. How much do map sizes vary? (affects CNN input design)
3. What does each defect pattern look like?
4. Does die yield differ by defect pattern?

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

from wafer import eda
from wafer.data import load_secom, load_wm811k

FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 3)

In [ ]:
df = load_wm811k()
print(f'{len(df):,} wafers')
df.head()

## 1. Labeled vs unlabeled, and class imbalance

In [ ]:
print(df['is_labeled'].value_counts(normalize=True).rename('share'))
summary = eda.label_summary(df)
summary

In [ ]:
ax = summary['count'].sort_values().plot.barh(figsize=(7, 4), logx=True)
ax.set_xlabel('wafers (log scale)')
ax.set_title('Labeled wafers per failure type')
plt.tight_layout(); plt.savefig(FIG_DIR / 'wm811k_class_counts.png', dpi=150)

> **Why it matters:** with `none` dominating, accuracy is misleading. Models will be compared on **macro-F1** and per-class recall, and training will use class weights / augmentation for rare classes such as `Near-full` and `Donut`.

## 2. Wafer map sizes

In [ ]:
shape_counts = df.groupby(['rows', 'cols']).size().sort_values(ascending=False)
print(f'{len(shape_counts)} distinct map shapes')
shape_counts.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df['n_dies'], bins=60)
ax.set_xlabel('dies per wafer'); ax.set_ylabel('wafers')
ax.set_title('Dies per wafer'); plt.tight_layout()

> **Why it matters:** maps come in many shapes, so the CNN needs a fixed input (resize to e.g. 64x64 with nearest-neighbour to keep die states discrete).

## 3. What each defect pattern looks like

In [ ]:
fig = eda.plot_wafer_grid(df[df['is_labeled']], per_class=5)
fig.savefig(FIG_DIR / 'wm811k_examples.png', dpi=150)

## 4. Defect pattern vs die yield

In [ ]:
eda.yield_by_failure_type(df)

In [ ]:
fig = eda.plot_yield_by_type(df)
fig.savefig(FIG_DIR / 'wm811k_yield_by_type.png', dpi=150)

## 5. Train / test split provided by the dataset

In [ ]:
lab = df[df['is_labeled']]
pd.crosstab(lab['failure_type'], lab['split'])

## Findings

_Fill in after running on the real data. Keep each point to one sentence with a number._

- Labeled share: …
- Largest / smallest class ratio: …
- Map shapes: …
- Lowest-yield patterns: …

**Decisions for step 2:** evaluation metric, input size, imbalance handling.